In [1]:
import torch 

In [ ]:
def custom_batch_norm(x,gamma,beta, running_mean, running_var, training, momentum=0.1, eps=1e-5):
    if training: 
        #
        mean = x.mean(dim=0)
        var = x.var(dim=0, unbiased=False)
        running_mean.copy_((1 - momentum) * running_mean + momentum * mean)
        running_var.copy_((1 - momentum) * running_var + momentum * var)

    else: 
        mean = running_mean
        var=running_var
    x_hat = (x - mean) / torch.sqrt(var + eps)

    out = gamma * x_hat + beta
    
    return out

In [ ]:
if __name__ == "__main__":
    torch.manual_seed(42)
    
   
    x_dummy = torch.randn(5, 3)
    
    
    gamma = torch.ones(3)
    beta = torch.zeros(3)
    

    running_mean = torch.zeros(3)
    running_var = torch.ones(3)
    
    custom_train_out = custom_batch_norm(
        x_dummy, gamma, beta, running_mean, running_var, training=True
    )
    
    official_bn = torch.nn.BatchNorm1d(num_features=3, eps=1e-5, momentum=0.1)
    official_bn.train()
    official_train_out = official_bn(x_dummy)
    
  
    assert torch.allclose(custom_train_out, official_train_out, atol=1e-5)
    print("✓ Training Mode outputs match PyTorch's native layer perfectly!")
    
  
    official_bn.eval() #
    custom_eval_out = custom_batch_norm(
        x_dummy, gamma, beta, running_mean, running_var, training=False
    )
    official_eval_out = official_bn(x_dummy)
    
    assert torch.allclose(custom_eval_out, official_eval_out, atol=1e-5)
    print("✓ Evaluation Mode outputs match PyTorch's native layer perfectly!")
    
  
    print("\nUpdated Running Mean:", running_mean)
    print("Updated Running Variance:", running_var)

✓ Training Mode outputs match PyTorch's native layer perfectly!


AssertionError: 